# Route labels — the golden set over our 50K composition

**Input.** `TargetComposition` — the 50,000 rows the composition package
selected. That is the dataset; nothing here invents queries.

**Output.** `data/route_labels/labels.parquet`: one row per labelled query
carrying the *route label* — which of `dense_only`, `pure_rrf`, `sparse_only`
retrieved best for it. Plus the losing routes' scores, the outcome shape, and
the composition's own `slice` / `checkable` / `label_lane` columns so results
can be read per slice.

**How the label is decided.** All three routes are run, each ranking is scored
by the router objective, argmax wins. No LLM is asked which route is better:
dense-vs-sparse depends on the corpus vocabulary and its IDF, neither of which
is in the query, so a model reading only the query is being asked to predict
something that isn't a function of its input.

**Honest coverage warning.** Labeling a row needs its dataset's corpus indexed
*and* its qrels. The registry cache is `[query_id, text]` for all 21 datasets —
doc_ids and qrels were dropped on ingest — so a lane opens only when a
per-dataset snapshot lands on disk. Snapshots exist for **beir-nfcorpus**
(323 rows) and, per SPEC d38, **msmarco-passage-dev** (7,697 of 15,678 — the
dev qrels cover 49.1%). `RouteLabels` reports every remaining row as
`unlabelled` rather than omitting it; lifting the registry's queries-only
restriction is the blocker for the rest.

**Prerequisites**

```bash
docker compose up -d          # local Qdrant on :6333
```

In [1]:
%load_ext autoreload
%autoreload 2

## 1 — Setup

In [25]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from qdrant_client import QdrantClient
from qdrant_client.models import Distance

load_dotenv(".env") or load_dotenv("../.env")

DENSE_MODEL, DENSE_SIZE = "BAAI/bge-small-en-v1.5", 384
SPARSE_MODEL = "Qdrant/bm25"

client = QdrantClient(
    url=os.getenv("QDRANT_URL", "http://localhost:6333"),
    api_key=os.getenv("QDRANT_API_KEY"),
)


def show(df: pd.DataFrame) -> None:
    display(Markdown(df.to_markdown(index=False)))


print("qdrant collections:", [c.name for c in client.get_collections().collections])

qdrant collections: ['bright-aops_routes', 'bright-leetcode_routes', 'bright-theoremqa-questions_routes', 'clerc_v5', 'composition_selection', 'crumb-clinical-trial_routes', 'crumb-code-retrieval_routes', 'crumb-legal-qa_routes', 'crumb-paper-retrieval_routes', 'crumb-set-operation-entity-retrieval_routes', 'crumb-stack-exchange_routes', 'crumb-theorem-retrieval_routes', 'crumb-tip-of-the-tongue_routes', 'limit_routes', 'msmarco_routes', 'musique', 'musique_colbert', 'nf', 'nfcorpus_routes', 'rarb-code_routes', 'rarb-math_routes', 'trec_dl']


## 2 — Load the composition

`TargetComposition().build()` returns the selection, reading `selection.parquet`
from disk when it exists — so this does not re-run the fill.

In [3]:
from composition import TargetComposition
from hybrid_search_rrf_dataset.labels import RouteLabels
from hybrid_search_rrf_dataset.objective import RouterObjective

selection = TargetComposition().build()
print(f"selection: {len(selection):,} rows x {len(selection.columns)} cols")
show(selection.head(5))

labels = RouteLabels(selection, objective=RouterObjective(min_relevance=1))
print("objective:", labels.objective.name)

selection: 50,000 rows x 7 cols


| dataset       | query_id   | checkable   | slice   | label_lane   | floors                                          | query                        |
|:--------------|:-----------|:------------|:--------|:-------------|:------------------------------------------------|:-----------------------------|
| beir-nfcorpus | PLAIN-1018 | True        | A       | qrels        | ['id:shape_guess' 'marker:acronym']             | DHA                          |
| beir-nfcorpus | PLAIN-1088 | True        | A       | qrels        | ['id:shape_guess' 'marker:acronym']             | ECMO                         |
| beir-nfcorpus | PLAIN-112  | True        | A       | qrels        | ['id:shape_guess' 'marker:acronym']             | Food Dyes and ADHD           |
| beir-nfcorpus | PLAIN-1320 | True        | A       | qrels        | ['id:shape_guess' 'marker:acronym']             | Harvard Physicians’ Study II |
| beir-nfcorpus | PLAIN-1398 | True        | A       | qrels        | ['id:number' 'id:shape_guess' 'marker:acronym'] | IGF-1                        |

objective: 0.7*HitRate@1+0.3*NDCG@10


## 3 — What can be labelled today

`unlabelled` means the row is in the composition but its dataset has no
indexed corpus or no qrels on disk. This table is the real progress bar for
the golden set, and it should be re-read after every dataset lands.

In [4]:
coverage = labels.coverage()
show(coverage)
print(f"selected {coverage.selected.sum():,} | "
      f"labelled {coverage.labelled.sum():,} | "
      f"unlabelled {coverage.unlabelled.sum():,} "
      f"({coverage.unlabelled.sum() / coverage.selected.sum() * 100:.1f}%)")

| dataset                              |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:-------------------------------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| msmarco-passage-dev                  |      15678 |       7697 |             0 |         7981 |            4006 |       3538 |        153 |
| rarb-code                            |       1117 |       1117 |             0 |            0 |             563 |          8 |        546 |
| beir-nfcorpus                        |        323 |        323 |             0 |            0 |             220 |         26 |         77 |
| orcas                                |      15744 |          0 |             0 |        15744 |               0 |          0 |          0 |
| rarb-math                            |       6276 |          0 |          6276 |         6276 |               0 |          0 |          0 |
| crumb-code-retrieval                 |       3665 |          0 |          3665 |         3665 |               0 |          0 |          0 |
| crumb-legal-qa                       |       3550 |          0 |          3550 |         3550 |               0 |          0 |          0 |
| quest                                |        928 |          0 |           928 |          928 |               0 |          0 |          0 |
| miracl-en-dev                        |        530 |          0 |           530 |          530 |               0 |          0 |          0 |
| crumb-set-operation-entity-retrieval |        423 |          0 |           423 |          423 |               0 |          0 |          0 |
| dbpedia-entity                       |        400 |          0 |           400 |          400 |               0 |          0 |          0 |
| limit                                |        344 |          0 |           344 |          344 |               0 |          0 |          0 |
| bright-theoremqa-questions           |        194 |          0 |           194 |          194 |               0 |          0 |          0 |
| bright-leetcode                      |        142 |          0 |           142 |          142 |               0 |          0 |          0 |
| crumb-tip-of-the-tongue              |        135 |          0 |           135 |          135 |               0 |          0 |          0 |
| crumb-clinical-trial                 |        113 |          0 |           113 |          113 |               0 |          0 |          0 |
| bright-aops                          |        111 |          0 |           111 |          111 |               0 |          0 |          0 |
| crumb-stack-exchange                 |        107 |          0 |           107 |          107 |               0 |          0 |          0 |
| trec-dl-2022                         |         79 |          0 |            16 |           79 |               0 |          0 |          0 |
| crumb-paper-retrieval                |         72 |          0 |            72 |           72 |               0 |          0 |          0 |
| crumb-theorem-retrieval              |         69 |          0 |            69 |           69 |               0 |          0 |          0 |

selected 50,000 | labelled 9,137 | unlabelled 40,863 (81.7%)


## 4 — Index one dataset's corpus

Starting with `beir-nfcorpus`: 3,633 documents, small enough to index in full
with no corpus sampling, and it ships human judgments so the labels are
checkable. It is also richly judged — 38.2 judged docs per query, and 300 of its
323 queries have two or more relevant documents, which is the only regime where
the choice of objective can change a label at all.

**Not** reusing the existing `nf` collection: it holds 33,633 points — 3,633
nfcorpus documents mixed with 30,000 trec-dl passages — so nfcorpus queries
would be scored against a corpus that is 90% unrelated.

Dense and sparse live as two named vector slots on one collection, which is what
lets a single Qdrant call fuse them. Idempotent: re-running skips the upload.

In [5]:
from hybrid_search_rrf_dataset.indexer import (
    CorpusDocument, CorpusIndexer, EmbeddingCache, EmbeddingConfig,
)
from hybrid_search_rrf_dataset.retrieval import SnapshotDataset

DATASET = "beir-nfcorpus"        # the composition's key
COLLECTION = "nfcorpus_routes"

source = SnapshotDataset("nfcorpus", path="data")   # written by RetrievalDataset.save()
corpus = source.corpus()

dense_cfg = EmbeddingConfig(
    name="dense_base", model_id=DENSE_MODEL, kind="dense",
    size=DENSE_SIZE, distance=Distance.COSINE,
)
sparse_cfg = EmbeddingConfig(name="sparse_base", model_id=SPARSE_MODEL, kind="sparse")

indexer = CorpusIndexer(
    client, COLLECTION,
    embeddings=[dense_cfg, sparse_cfg],
    cache=EmbeddingCache("./.embedding_cache"),
)
indexer.ensure_collection()

if client.count(COLLECTION, exact=True).count >= len(corpus):
    print(f"{COLLECTION}: already indexed — skipping upload")
else:
    indexer.upload([CorpusDocument(**r) for r in corpus.to_dict("records")], batch_size=64)
print(f"{COLLECTION}: {client.count(COLLECTION, exact=True).count:,} points")

nfcorpus_routes: already indexed — skipping upload
nfcorpus_routes: 3,633 points


## 5 — The three routes

`DenseOnlyStrategy` and `SparseOnlyStrategy` keep their raw scores;
`PureRRFStrategy` is Qdrant-native RRF, matching production's `Fusion::Rrf`.

RRF fuses by *rank position*, which is exactly why it can lose on top-1: a doc
ranked first by dense and 40th by sparse loses to one ranked third by both.

In [6]:
from hybrid_search_rrf_dataset.fusion import (
    DenseOnlyStrategy, PureRRFStrategy, SparseOnlyStrategy,
)

args = (client, COLLECTION, dense_cfg, sparse_cfg)
dense, hybrid, sparse = (
    DenseOnlyStrategy(*args), PureRRFStrategy(*args), SparseOnlyStrategy(*args)
)

probe = labels.rows_for(DATASET)["query"].iloc[0]
print(f"probe (a real selection row): {probe!r}\n")
for s in (dense, hybrid, sparse):
    top = list(s.rank(probe).items())[:3]
    print(f"  {s.name:12s} " + "  ".join(f"{d}={v:.3f}" for d, v in top))

probe (a real selection row): 'DHA'

  dense_only   MED-5095=0.735  MED-5091=0.718  MED-4936=0.714
  pure_rrf     MED-5095=1.000  MED-4936=0.583  MED-5091=0.533
  sparse_only  MED-5095=3.486  MED-4936=3.465  MED-839=3.449


## 6 — Label this dataset's selection rows

`RouteLabels.label` narrows the source to the composition's query ids via
`QuerySubset`, runs `GoldenRoutingBuilder`, and merges the result into
`labels.parquet` — replacing only this dataset's rows.

The objective is `0.7·HitRate@1 + 0.3·NDCG@10`. Because the hit weight exceeds
the NDCG weight the score ranges are disjoint (rank-1 hit ⇒ ≥0.700, miss ⇒
≤0.300), so it is lexicographic: top-1 decides, NDCG@10 only breaks ties within
each group. `min_relevance=1` because nfcorpus grades are 1 and 2 only.

In [7]:
labelled = labels.label(source, dense, hybrid, sparse, dataset=DATASET)
print(f"labelled {len(labelled):,} rows -> {labels.labels_path}")
show(labelled.head(8).round(3))

labelled 323 rows -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/route_labels/labels.parquet


| dataset       | query_id   | route       | query                                                    |   score |   score_dense_only |   score_pure_rrf |   score_sparse_only | shape         | metric_name               |   min_relevance | slice   | checkable   | label_lane   |
|:--------------|:-----------|:------------|:---------------------------------------------------------|--------:|-------------------:|-----------------:|--------------------:|:--------------|:--------------------------|----------------:|:--------|:------------|:-------------|
| beir-nfcorpus | PLAIN-2    | pure_rrf    | Do Cholesterol Statin Drugs Cause Breast Cancer?         |   0.946 |              0.933 |            0.946 |               0.916 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-12   | pure_rrf    | Exploiting Autophagy to Live Longer                      |   0.773 |              0     |            0.773 |               0.773 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-23   | dense_only  | How to Reduce Exposure to Alkylphenols Through Your Diet |   0.847 |              0.847 |            0.08  |               0     | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-33   | dense_only  | What’s Driving America’s Obesity Problem?                |   0.834 |              0.834 |            0.116 |               0.063 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-44   | pure_rrf    | Who Should be Careful About Curcumin?                    |   0.78  |              0.776 |            0.78  |               0.053 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-56   | dense_only  | Foods for Glaucoma                                       |   0.903 |              0.903 |            0.89  |               0.853 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-68   | dense_only  | What is Actually in Chicken Nuggets?                     |   0.914 |              0.914 |            0.907 |               0.859 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| beir-nfcorpus | PLAIN-78   | sparse_only | What Do Meat Purge and Cola Have in Common?              |   0.025 |              0     |            0     |               0.025 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |

## 7 — Outcome shapes

Only one of the three shapes teaches the router about dense-vs-sparse.

| shape | meaning |
| --- | --- |
| `routes_differ` | the quality signal — the trainable set |
| `all_tied` | any route works; serve the cheapest. Signal for the *speed* goal |
| `all_zero` | nothing relevant found by any route — unanswerable, and **no valid label exists**. The argmax falls through to list order and reports `dense_only`, which is fabricated |

Read per composition slice: A/B are span-evidence rows, C is the stat strata,
D is the feature-blind draw.

In [8]:
shapes = (labelled.groupby(["slice", "shape"]).size()
          .unstack(fill_value=0))
shapes["total"] = shapes.sum(axis=1)
show(shapes.reset_index())

overall = labelled["shape"].value_counts().rename_axis("shape").reset_index(name="rows")
overall["share"] = (overall.rows / len(labelled) * 100).round(1).astype(str) + "%"
show(overall)

signal = labelled[labelled["shape"] == "routes_differ"]
dist = signal.route.value_counts().rename_axis("route").reset_index(name="rows")
dist["share"] = (dist.rows / len(signal) * 100).round(1).astype(str) + "%"
print(f"route distribution over the {len(signal):,} trainable rows:")
show(dist)
print(f"majority-class baseline: {dist.rows.iloc[0] / len(signal) * 100:.0f}% "
      f"(always predict {dist.route.iloc[0]})")

| slice   |   all_tied |   all_zero |   routes_differ |   total |
|:--------|-----------:|-----------:|----------------:|--------:|
| A       |          2 |          7 |              19 |      28 |
| B       |          0 |          0 |               8 |       8 |
| C       |         24 |         70 |             193 |     287 |

| shape         |   rows | share   |
|:--------------|-------:|:--------|
| routes_differ |    220 | 68.1%   |
| all_zero      |     77 | 23.8%   |
| all_tied      |     26 | 8.0%    |

route distribution over the 220 trainable rows:


| route       |   rows | share   |
|:------------|-------:|:--------|
| dense_only  |    107 | 48.6%   |
| pure_rrf    |     64 | 29.1%   |
| sparse_only |     49 | 22.3%   |

majority-class baseline: 49% (always predict dense_only)


## 8 — Constant-route baselines

The bar a router must clear is **the best constant route**, not random. A
candidate can post respectable regret against the oracle and still lose to one
line of code, so these belong in every comparison.

In [9]:
from hybrid_search_rrf_dataset.evaluation import compare
from hybrid_search_rrf_dataset.golden import BaselineBuilder, GoldenRoutingBuilder
from hybrid_search_rrf_dataset.retrieval import QuerySubset

subset = QuerySubset(source, labels.rows_for(DATASET)["query_id"])
oracle = GoldenRoutingBuilder(
    dense, hybrid, sparse, objective=labels.objective
).build_or_load(subset, path=f"data/route_labels/{DATASET}_oracle")

rows = []
for strategy in (dense, hybrid, sparse):
    const = BaselineBuilder(strategy, objective=labels.objective).build_or_load(
        subset, path=f"data/route_labels/{DATASET}_const_{strategy.name}"
    )
    s = compare(oracle, const)
    rows.append({
        "always pick": str(strategy.name),
        "mean score": round(s.mean_metric_candidate, 3),
        "oracle ceiling": round(s.mean_metric_golden, 3),
        "mean regret": round(s.mean_regret, 3),
        "p90 regret": round(s.p90_regret, 3),
        "reaches oracle": f"{s.oracle_hit_rate_pct:.0f}%",
        "route agreement": f"{s.route_agreement_pct:.0f}%",
    })
show(pd.DataFrame(rows))

| always pick   |   mean score |   oracle ceiling |   mean regret |   p90 regret | reaches oracle   | route agreement   |
|:--------------|-------------:|-----------------:|--------------:|-------------:|:-----------------|:------------------|
| dense_only    |        0.403 |            0.504 |         0.102 |        0.727 | 65%              | 65%               |
| pure_rrf      |        0.443 |            0.504 |         0.062 |        0.047 | 55%              | 20%               |
| sparse_only   |        0.389 |            0.504 |         0.116 |        0.739 | 53%              | 15%               |

## 9 — Objective sensitivity, at zero retrieval cost

`route_rankings` holds each route's top-10 doc ids and the judgments live in
`QrelStore`, so any metric depending only on the *order* of the top 10 can be
recomputed without touching Qdrant. That is what makes the objective a
reversible decision instead of a one-way door.

We synthesize descending scores from the stored order — HitRate@1 and NDCG@10
depend only on that order, so the recomputation is exact. It cannot evaluate a
metric needing rank 11+ or the raw retrieval scores.

In [10]:
from hybrid_search_rrf_dataset.objective import NDCGObjective
from hybrid_search_rrf_dataset.qrels import QrelStore

lookup = QrelStore.from_dataset(source).lookup(source.name)


def relabel(objective) -> pd.Series:
    picks = []
    for row in oracle:
        gold = lookup.get(row.query_id, {})
        scored = {
            route: objective.assess(
                {d: 1.0 / (i + 1) for i, d in enumerate(ids)}, gold
            )[0]
            for route, ids in row.route_rankings.items()
        }
        picks.append(max(scored, key=lambda route: scored[route]))
    return pd.Series(picks)


shipped = pd.Series([str(r.strategy_name) for r in oracle])
variants = {
    "0.7·HR@1 + 0.3·NDCG@10  (shipped)": RouterObjective(min_relevance=1),
    "0.5·HR@1 + 0.5·NDCG@10": RouterObjective(hit_weight=0.5, ndcg_weight=0.5, min_relevance=1),
    "bare NDCG@10 (no top-1 term)": NDCGObjective(min_relevance=1),
    "shipped, stricter min_relevance=2": RouterObjective(min_relevance=2),
}

rows = []
for name, objective in variants.items():
    picks = relabel(objective)
    rows.append({
        "objective": name,
        "flips": int((picks != shipped).sum()),
        "flip rate": f"{(picks != shipped).mean() * 100:.1f}%",
        **{f"picks {r}": int((picks == r).sum())
           for r in ("dense_only", "pure_rrf", "sparse_only")},
    })
show(pd.DataFrame(rows))

| objective                         |   flips | flip rate   |   picks dense_only |   picks pure_rrf |   picks sparse_only |
|:----------------------------------|--------:|:------------|-------------------:|-----------------:|--------------------:|
| 0.7·HR@1 + 0.3·NDCG@10  (shipped) |       0 | 0.0%        |                211 |               63 |                  49 |
| 0.5·HR@1 + 0.5·NDCG@10            |       0 | 0.0%        |                211 |               63 |                  49 |
| bare NDCG@10 (no top-1 term)      |       3 | 0.9%        |                212 |               62 |                  49 |
| shipped, stricter min_relevance=2 |      89 | 27.6%       |                293 |               16 |                  14 |

## 10 — Where the golden set stands

Re-read coverage now that one dataset is labelled, and project the trainable
yield. The projection assumes other datasets behave like this one, which they
will not — nfcorpus is a single 3,633-document medical corpus, unusually
richly judged. Treat it as an order of magnitude, not a forecast.

In [11]:
coverage = labels.coverage()
show(coverage[coverage.labelled > 0])

done = labels.load()
trainable = (done["shape"] == "routes_differ").mean()
print(f"labelled so far:        {len(done):,} of {len(selection):,} rows")
print(f"trainable share:        {trainable * 100:.1f}%")
print(f"projected yield at 50K: ~{int(trainable * len(selection)):,} rows "
      f"(extrapolated from one dataset — see caveat above)")
print()
print("next unblock, largest first:")
show(coverage[coverage.labelled == 0].head(5)[["dataset", "selected", "unlabelled"]])

| dataset             |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:--------------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| msmarco-passage-dev |      15678 |       7697 |             0 |         7981 |            4006 |       3538 |        153 |
| rarb-code           |       1117 |       1117 |             0 |            0 |             563 |          8 |        546 |
| beir-nfcorpus       |        323 |        323 |             0 |            0 |             220 |         26 |         77 |

labelled so far:        9,137 of 50,000 rows
trainable share:        52.4%
projected yield at 50K: ~26,206 rows (extrapolated from one dataset — see caveat above)

next unblock, largest first:


| dataset              |   selected |   unlabelled |
|:---------------------|-----------:|-------------:|
| orcas                |      15744 |        15744 |
| rarb-math            |       6276 |         6276 |
| crumb-code-retrieval |       3665 |         3665 |
| crumb-legal-qa       |       3550 |         3550 |
| quest                |        928 |          928 |

## 11 — msmarco-passage-dev: the composition's largest lane (SPEC d38)

15,678 composition rows — 31% of the 50K. The local dev qrels cover 7,697 of
them (49.1%); the rest stay `unlabelled` in coverage — a gap to report, never
a licence to substitute other queries. The gap is MS MARCO's own: the full
dev set ships 101,093 queries but judgments were released for only 55,578
(55%), and the fill drew judgment-blind — the composition's `checkable=True`
was assigned per-dataset, not per-query. Median **one** judged passage per
query against nfcorpus's 16, so this lane sits at the opposite end of the
judgment-density axis: the regime where two different top-10 lists cannot
both be right.

The corpus recipe (d38c): every judged-relevant passage for the selected
queries is force-included, then padded with uniform-random passages from the
full 8.8M collection to **100,000** total, fixed seed. Uniform sampling
preserves the collection's vocabulary/IDF profile — the d37(g) fix: trec-dl's
judged-docs-only corpus was near-all answers, which inflates dense and
starves sparse.

Materialization is one-time and local (the ir_datasets collection is already
on disk; first `docs_store` access builds its index). Re-runs read the
snapshot back via `SnapshotDataset`.

In [12]:
from pathlib import Path

from hybrid_search_rrf_dataset.retrieval import MSMarcoDev

MS_DATASET = "msmarco-passage-dev"      # composition key == source name here
MS_COLLECTION = "msmarco_routes"

if not (Path("data") / MS_DATASET / "corpus.parquet").exists():
    ms = MSMarcoDev(
        query_ids=labels.rows_for(MS_DATASET)["query_id"],
        corpus_size=100_000,
        seed=0,                          # d38(c): fixed seed, recipe is a parameter
    )
    ms.materialize()
    ms.save("data")

ms_source = SnapshotDataset(MS_DATASET, path="data")
ms_corpus, ms_queries, ms_qrels = ms_source.corpus(), ms_source.queries(), ms_source.qrels()
print(f"corpus  {len(ms_corpus):,} passages "
      f"({ms_qrels.doc_id.nunique():,} judged-relevant, rest uniform distractors)")
print(f"queries {len(ms_queries):,} of {len(labels.rows_for(MS_DATASET)):,} composition rows "
      f"({len(ms_queries) / len(labels.rows_for(MS_DATASET)) * 100:.1f}% have dev qrels)")
print(f"qrels   {len(ms_qrels):,} judgments, grades {sorted(ms_qrels.relevance.unique())}")

corpus  100,000 passages (8,219 judged-relevant, rest uniform distractors)
queries 7,697 of 15,678 composition rows (49.1% have dev qrels)
qrels   8,228 judgments, grades [np.int64(1)]


## 12 — Index the 100K corpus

Same two named vector slots as every route collection. The one-time cost is
the dense pass over 100K passages (order of an hour on this machine); the
embedding cache makes re-runs cheap, and the upload skips when the collection
is already full.

In [13]:
ms_indexer = CorpusIndexer(
    client, MS_COLLECTION,
    embeddings=[dense_cfg, sparse_cfg],
    cache=EmbeddingCache("./.embedding_cache"),
)
ms_indexer.ensure_collection()

if client.count(MS_COLLECTION, exact=True).count >= len(ms_corpus):
    print(f"{MS_COLLECTION}: already indexed — skipping upload")
else:
    ms_indexer.upload(
        [CorpusDocument(**r) for r in ms_corpus.to_dict("records")], batch_size=64
    )
print(f"{MS_COLLECTION}: {client.count(MS_COLLECTION, exact=True).count:,} points")

msmarco_routes: already indexed — skipping upload
msmarco_routes: 100,000 points


## 13 — Label the lane

Same argmax rule as the nfcorpus anchor (d38e), so the two datasets differ by
exactly one variable — the index they are scored on. `RouteLabels.label`
narrows the 15,678 selection rows to the snapshot's queries via `QuerySubset`
and merges only this dataset's rows into `labels.parquet`. `min_relevance=1`
holds: the dev qrels are binary.

~7,700 queries × 3 routes; at nfcorpus throughput this is on the order of ten
minutes against local Qdrant.

In [14]:
ms_args = (client, MS_COLLECTION, dense_cfg, sparse_cfg)
ms_dense, ms_hybrid, ms_sparse = (
    DenseOnlyStrategy(*ms_args), PureRRFStrategy(*ms_args), SparseOnlyStrategy(*ms_args)
)

ms_labelled = labels.label(ms_source, ms_dense, ms_hybrid, ms_sparse, dataset=MS_DATASET)
print(f"labelled {len(ms_labelled):,} rows -> {labels.labels_path}")
show(ms_labelled.head(8).round(3))

labelled 7,697 rows -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/route_labels/labels.parquet


| dataset             |   query_id | route      | query                                                    |   score |   score_dense_only |   score_pure_rrf |   score_sparse_only | shape         | metric_name               |   min_relevance | slice   | checkable   | label_lane   |
|:--------------------|-----------:|:-----------|:---------------------------------------------------------|--------:|-------------------:|-----------------:|--------------------:|:--------------|:--------------------------|----------------:|:--------|:------------|:-------------|
| msmarco-passage-dev |          2 | dense_only | Androgen receptor define                                 |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| msmarco-passage-dev |    1048600 | dense_only | what is patricia cornwell's latest book                  |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | A       | True        | qrels        |
| msmarco-passage-dev |     524332 | dense_only | treating tension headaches without medication            |   0.189 |              0.189 |             0.15 |               0     | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | A       | True        | qrels        |
| msmarco-passage-dev |    1048663 | dense_only | what is palm harbor florida                              |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| msmarco-passage-dev |     786568 | dense_only | what is price of pressure treated lumber 2x6x8           |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| msmarco-passage-dev |     786598 | dense_only | what is primary and non-contributory under the liability |   1     |              1     |             1    |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |
| msmarco-passage-dev |    1048836 | dense_only | who plays velma in scooby doo 2                          |   1     |              1     |             1    |               0.189 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | A       | True        | qrels        |
| msmarco-passage-dev |    1048846 | dense_only | what is option button style ?                            |   1     |              1     |             1    |               0.189 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | C       | True        | qrels        |

## 14 — Outcome shapes on a realistic index

Same three shapes as §7, now on a corpus that is 92% distractors. With a
median of **one** judged passage per query, `all_tied` requires all three
routes to place that same passage at the same effective rank, and
`routes_differ` means at least one route actually found it while another
did not — a much harder tie regime than nfcorpus's.

In [15]:
ms_shapes = (ms_labelled.groupby(["slice", "shape"]).size()
             .unstack(fill_value=0))
ms_shapes["total"] = ms_shapes.sum(axis=1)
show(ms_shapes.reset_index())

overall = ms_labelled["shape"].value_counts().rename_axis("shape").reset_index(name="rows")
overall["share"] = (overall.rows / len(ms_labelled) * 100).round(1).astype(str) + "%"
show(overall)

signal = ms_labelled[ms_labelled["shape"] == "routes_differ"]
dist = signal.route.value_counts().rename_axis("route").reset_index(name="rows")
dist["share"] = (dist.rows / len(signal) * 100).round(1).astype(str) + "%"
print(f"route distribution over the {len(signal):,} trainable rows:")
show(dist)
print(f"majority-class baseline: {dist.rows.iloc[0] / len(signal) * 100:.0f}% "
      f"(always predict {dist.route.iloc[0]})")

| slice   |   all_tied |   all_zero |   routes_differ |   total |
|:--------|-----------:|-----------:|----------------:|--------:|
| A       |        818 |         52 |             978 |    1848 |
| B       |        190 |         11 |             206 |     407 |
| C       |       1710 |         62 |            1971 |    3743 |
| D       |        820 |         28 |             851 |    1699 |

| shape         |   rows | share   |
|:--------------|-------:|:--------|
| routes_differ |   4006 | 52.0%   |
| all_tied      |   3538 | 46.0%   |
| all_zero      |    153 | 2.0%    |

route distribution over the 4,006 trainable rows:


| route       |   rows | share   |
|:------------|-------:|:--------|
| dense_only  |   3333 | 83.2%   |
| pure_rrf    |    474 | 11.8%   |
| sparse_only |    199 | 5.0%    |

majority-class baseline: 83% (always predict dense_only)


## 15 — Where the golden set stands, two datasets in

The open label-form question (argmax one-hot vs the per-route score vector,
TODOS) turns on the **margin** — winner's score minus runner-up's. A fat
margin is a fact about retrieval; a thin one is a coin toss the objective
happened to break, and it flips when the encoder or corpus changes.
`decisive` counts rows with margin ≥ 0.06 *and* a rank-1 hit — roomier than
any NDCG-tail wiggle, and excluding "least bad" wins where every route missed.

nfcorpus margins were thin because 86% of its corpus is judged relevant to
*something* — two disjoint top-10s can both be right. msmarco's
median-1-relevant regime is the counter-test: either a route surfaced the one
judged passage or it scored zero, so ties require actually retrieving the
same passage at the same rank.

In [16]:
SCORES = ["score_dense_only", "score_pure_rrf", "score_sparse_only"]

done = labels.load()
done["margin"] = done[SCORES].max(axis=1) - done[SCORES].apply(
    lambda r: sorted(r)[-2], axis=1
)
done["hit"] = done[SCORES].max(axis=1) >= 0.7

rows = []
for ds, group in done.groupby("dataset"):
    differ = group[group["shape"] == "routes_differ"]
    decisive = (differ.margin >= 0.06) & differ.hit
    rows.append({
        "dataset": ds,
        "labelled": len(group),
        "routes_differ": len(differ),
        "median margin": round(differ.margin.median(), 3),
        "p90 margin": round(differ.margin.quantile(0.9), 3),
        "decisive": int(decisive.sum()),
        "decisive share": f"{decisive.mean() * 100:.0f}%",
    })
show(pd.DataFrame(rows))

coverage = labels.coverage()
show(coverage[coverage.labelled > 0])
print(f"labelled {coverage.labelled.sum():,} of {coverage.selected.sum():,} "
      f"({coverage.labelled.sum() / coverage.selected.sum() * 100:.1f}%)")

| dataset             |   labelled |   routes_differ |   median margin |   p90 margin |   decisive | decisive share   |
|:--------------------|-----------:|----------------:|----------------:|-------------:|-----------:|:-----------------|
| beir-nfcorpus       |        323 |             220 |           0.011 |        0.707 |         28 | 13%              |
| msmarco-passage-dev |       7697 |            4006 |           0     |        0.811 |        927 | 23%              |
| rarb-code           |       1117 |             563 |           0.06  |        0.811 |        132 | 23%              |

| dataset             |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:--------------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| msmarco-passage-dev |      15678 |       7697 |             0 |         7981 |            4006 |       3538 |        153 |
| rarb-code           |       1117 |       1117 |             0 |            0 |             563 |          8 |        546 |
| beir-nfcorpus       |        323 |        323 |             0 |            0 |             220 |         26 |         77 |

labelled 9,137 of 50,000 (18.3%)


## 16 — Pass 1: qrels for every lane (SPEC d39)

The route distribution above (84% dense over decisive rows) is an artifact of
*which* lanes are labelled — two natural-language lanes where the dense
encoder is at home. The sparse/hybrid signal lives in the technical and
entity lanes, and this pass opens them: `LANES` maps every QQ composition
dataset to its acquisition owner, and the loop below fetches **queries +
qrels only** (corpus-pending snapshots — corpora follow in pass 2, wave by
wave). Running the cell is the user-initiated network fetch; each lane is hit
once, then everything reads from disk.

Sizes and quirks to expect:

- **dbpedia-entity** — first access downloads the ~2GB BEIR zip (corpus
  included, so pass 2 pays nothing new). The one graded lane here (0/1/2).
- **crumb-*** — the qrels config name is the plan's one guessed dialect; a
  loud schema guard names the real fields if the guess is wrong, and the fix
  is a two-string edit in `CrumbLane`.
- **bright-*** — `gold_ids` become qrels; `excluded_ids` persist to
  `excluded.parquet` per lane (applied at scoring time in the labeling
  stage — a doc excluded for one query can be gold for another).
- **quest** — ids are synthesized `{split}-{index}` in stream order, the
  registry's own convention, so they join the composition's ids exactly.

The coverage re-read then shows the pipeline per lane: `unlabelled` →
`qrels_ready` → `labelled`. Expect qrels_ready ≪ selected for some lanes —
msmarco's 49.1% taught us that declared QQ grounding does not guarantee
per-query coverage.

In [17]:
from hybrid_search_rrf_dataset.lanes import LANES
from pathlib import Path


for key, lane in LANES.items():
    target = Path("data") / lane.source.name / "qrels.parquet"
    if target.exists():
        print(f"{key:42s} qrels on disk — skip")
        continue
    lane.source.load_metadata()
    lane.source.save_metadata("data")
    q, r = lane.source.queries(), lane.source.qrels()
    print(f"{key:42s} queries {len(q):,}  qrels {len(r):,}  "
          f"grades {sorted(r.relevance.unique())}")

beir-nfcorpus                              qrels on disk — skip
msmarco-passage-dev                        qrels on disk — skip
trec-dl-2022                               qrels on disk — skip
rarb-math                                  qrels on disk — skip
rarb-code                                  qrels on disk — skip
bright-aops                                qrels on disk — skip
bright-leetcode                            qrels on disk — skip
bright-theoremqa-questions                 qrels on disk — skip
crumb-clinical-trial                       qrels on disk — skip
crumb-code-retrieval                       qrels on disk — skip
crumb-legal-qa                             qrels on disk — skip
crumb-paper-retrieval                      qrels on disk — skip
crumb-set-operation-entity-retrieval       qrels on disk — skip
crumb-stack-exchange                       qrels on disk — skip
crumb-theorem-retrieval                    qrels on disk — skip
crumb-tip-of-the-tongue                 

In [18]:
coverage = labels.coverage()
show(coverage)
print(f"selected {coverage.selected.sum():,} | "
      f"labelled {coverage.labelled.sum():,} | "
      f"qrels_ready {coverage.qrels_ready.sum():,} | "
      f"unlabelled {coverage.unlabelled.sum():,}")

| dataset                              |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:-------------------------------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| msmarco-passage-dev                  |      15678 |       7697 |             0 |         7981 |            4006 |       3538 |        153 |
| rarb-code                            |       1117 |       1117 |             0 |            0 |             563 |          8 |        546 |
| beir-nfcorpus                        |        323 |        323 |             0 |            0 |             220 |         26 |         77 |
| orcas                                |      15744 |          0 |             0 |        15744 |               0 |          0 |          0 |
| rarb-math                            |       6276 |          0 |          6276 |         6276 |               0 |          0 |          0 |
| crumb-code-retrieval                 |       3665 |          0 |          3665 |         3665 |               0 |          0 |          0 |
| crumb-legal-qa                       |       3550 |          0 |          3550 |         3550 |               0 |          0 |          0 |
| quest                                |        928 |          0 |           928 |          928 |               0 |          0 |          0 |
| miracl-en-dev                        |        530 |          0 |           530 |          530 |               0 |          0 |          0 |
| crumb-set-operation-entity-retrieval |        423 |          0 |           423 |          423 |               0 |          0 |          0 |
| dbpedia-entity                       |        400 |          0 |           400 |          400 |               0 |          0 |          0 |
| limit                                |        344 |          0 |           344 |          344 |               0 |          0 |          0 |
| bright-theoremqa-questions           |        194 |          0 |           194 |          194 |               0 |          0 |          0 |
| bright-leetcode                      |        142 |          0 |           142 |          142 |               0 |          0 |          0 |
| crumb-tip-of-the-tongue              |        135 |          0 |           135 |          135 |               0 |          0 |          0 |
| crumb-clinical-trial                 |        113 |          0 |           113 |          113 |               0 |          0 |          0 |
| bright-aops                          |        111 |          0 |           111 |          111 |               0 |          0 |          0 |
| crumb-stack-exchange                 |        107 |          0 |           107 |          107 |               0 |          0 |          0 |
| trec-dl-2022                         |         79 |          0 |            16 |           79 |               0 |          0 |          0 |
| crumb-paper-retrieval                |         72 |          0 |            72 |           72 |               0 |          0 |          0 |
| crumb-theorem-retrieval              |         69 |          0 |            69 |           69 |               0 |          0 |          0 |

selected 50,000 | labelled 9,137 | qrels_ready 17,075 | unlabelled 40,863


## 17 — Pass 2, wave 1: the technical lanes (SPEC d39 e/f, computed 20/80)

Fourteen lanes, most important first. Per lane the loop does what §11–§13
did for msmarco, via the shared machinery: hydrate qrels from the pass-1
snapshot (never re-fetching) → `materialize()` sizes the corpus with
**`CorpusRecipe`** — `target = clamp(relevant / answer_share, floor,
ceiling)` with `answer_share=0.2, floor=10K, ceiling=100K` — computed from
the lane's own qrels at run time, never hand-picked. Every judged-relevant
doc is force-included; seeded uniform reservoir distractors fill to the
target. Then index into `<lane>_routes` → label with the same argmax
objective. Idempotent at every step.

The recipe's three parameters are the discussion surface: `answer_share`
because the label is an argmax over three routes on the *same* index and the
corpus-room diagnostic showed 9.3× corpus growth flips only 13.6% of labels
(`all_zero` only grows — distractor mass buys difficulty, not route signal);
`floor` so no corpus drops under the strategies' fetch depth and turns
ranking trivial; `ceiling` so answer-heavy lanes don't balloon (clinical
computes to 198K unbounded). Exceptions are pinned in `LANES` with their
reasons: crumb-code 120K (108,782 relevant — median 23 relevant docs per
query by benchmark design — exceed the ceiling; floor-plus-pad, since the
hard confusables are in the forced set either way), limit full 50K (the
recipe would compute the floor and delete the stress test), rarb-code at
its already-paid 100K.

BRIGHT lanes score with `excluded_ids` applied **per query at scoring
time** — measured: 6,393 excluded pairs are another query's gold, so
corpus-level dropping would corrupt other rows. The fetch depth (1,000)
refills the top-10 after the drop.

quest, dbpedia-entity, miracl-en-dev are wave 2 — same loop, different list.

In [21]:
from hybrid_search_rrf_dataset.lanes import LANES

WAVE1 = [
    # most important first — rows per embedding-hour
    "rarb-math", "crumb-legal-qa", "rarb-code", "limit",
    "crumb-stack-exchange", "crumb-theorem-retrieval",
    "crumb-code-retrieval",
    "crumb-set-operation-entity-retrieval", "bright-theoremqa-questions",
    "bright-leetcode", "bright-aops", "crumb-tip-of-the-tongue",
    "crumb-clinical-trial", "crumb-paper-retrieval",
]

for key in WAVE1:
    lane = LANES[key]
    name = lane.source.name

    if not (Path("data") / name / "corpus.parquet").exists():
        lane.source.hydrate("data") or lane.source.load_metadata()
        lane.source.corpus_target = lane.corpus_target   # exceptions only; None = recipe
        lane.source.materialize()
        lane.source.save("data")

    src = SnapshotDataset(name, path="data")
    corpus = src.corpus()
    collection = f"{name}_routes"

    lane_indexer = CorpusIndexer(
        client, collection,
        embeddings=[dense_cfg, sparse_cfg],
        cache=EmbeddingCache("./.embedding_cache"),
    )
    lane_indexer.ensure_collection()
    if client.count(collection, exact=True).count < len(corpus):
        lane_indexer.upload(
            [CorpusDocument(**r) for r in corpus.to_dict("records")], batch_size=64
        )

    strat_args = (client, collection, dense_cfg, sparse_cfg)
    out = labels.label(
        src,
        DenseOnlyStrategy(*strat_args),
        PureRRFStrategy(*strat_args),
        SparseOnlyStrategy(*strat_args),
        dataset=key,
    )
    shapes = out["shape"].value_counts()
    print(f"{key:42s} corpus {len(corpus):>7,}  labelled {len(out):>6,}  "
          f"differ {shapes.get('routes_differ', 0):,} | tied {shapes.get('all_tied', 0):,} "
          f"| zero {shapes.get('all_zero', 0):,}")

rarb-math                                  corpus  31,595  labelled  6,276  differ 3,191 | tied 2,116 | zero 969
crumb-legal-qa                             corpus  10,000  labelled  3,550  differ 2,174 | tied 84 | zero 1,292
rarb-code                                  corpus 100,000  labelled  1,117  differ 563 | tied 8 | zero 546
limit                                      corpus  50,000  labelled    344  differ 317 | tied 0 | zero 27
crumb-stack-exchange                       corpus  10,000  labelled    107  differ 56 | tied 9 | zero 42
crumb-theorem-retrieval                    corpus  10,000  labelled     69  differ 20 | tied 0 | zero 49


goldenroutingbuilder:crumb-code-retrieval: 100%|██████████| 3665/3665 [09:03<00:00,  6.74it/s]  


crumb-code-retrieval                       corpus 119,976  labelled  3,665  differ 903 | tied 24 | zero 2,738


materialize:crumb-set-operation-entity-retrieval: 651704doc [01:49, 5955.12doc/s] 
embed:sparse_base: 100%|██████████| 57730/57730 [00:10<00:00, 5476.16it/s]
goldenroutingbuilder:crumb-set-operation-entity-retrieval: 100%|██████████| 423/423 [00:40<00:00, 10.37it/s]


crumb-set-operation-entity-retrieval       corpus  57,730  labelled    423  differ 326 | tied 3 | zero 94


...

crumb-tip-of-the-tongue                    corpus  10,000  labelled    135  differ 66 | tied 1 | zero 68


materialize:crumb-clinical-trial: 914628doc [03:03, 4992.94doc/s]
goldenroutingbuilder:crumb-clinical-trial: 100%|██████████| 113/113 [00:22<00:00,  5.07it/s]


crumb-clinical-trial                       corpus 100,000  labelled    113  differ 111 | tied 1 | zero 1


materialize:crumb-paper-retrieval: 363133doc [01:12, 5037.51doc/s]
goldenroutingbuilder:crumb-paper-retrieval: 100%|██████████| 72/72 [00:13<00:00,  5.42it/s]

crumb-paper-retrieval                      corpus  28,495  labelled     72  differ 72 | tied 0 | zero 0


In [23]:
coverage = labels.coverage()
show(coverage[coverage.labelled > 0])
print(f"labelled {coverage.labelled.sum():,} of {coverage.selected.sum():,} | "
      f"qrels_ready {coverage.qrels_ready.sum():,}")

done = labels.load()
done["margin"] = done[SCORES].max(axis=1) - done[SCORES].apply(
    lambda r: sorted(r)[-2], axis=1
)
done["hit"] = done[SCORES].max(axis=1) >= 0.7

rows = []
for ds, group in done.groupby("dataset"):
    differ = group[group["shape"] == "routes_differ"]
    decisive = (differ.margin >= 0.06) & differ.hit
    dist = differ.route.value_counts()
    rows.append({
        "dataset": ds,
        "labelled": len(group),
        "routes_differ": len(differ),
        "decisive": int(decisive.sum()),
        "decisive share": f"{decisive.mean() * 100:.0f}%",
        "median margin": round(differ.margin.median(), 3) if len(differ) else None,
        "top route": dist.index[0] if len(dist) else None,
        "top share": f"{dist.iloc[0] / len(differ) * 100:.0f}%" if len(differ) else None,
    })
show(pd.DataFrame(rows).sort_values("labelled", ascending=False))

| dataset                              |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:-------------------------------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| msmarco-passage-dev                  |      15678 |       7697 |             0 |         7981 |            4006 |       3538 |        153 |
| rarb-math                            |       6276 |       6276 |             0 |            0 |            3191 |       2116 |        969 |
| crumb-code-retrieval                 |       3665 |       3665 |             0 |            0 |             903 |         24 |       2738 |
| crumb-legal-qa                       |       3550 |       3550 |             0 |            0 |            2174 |         84 |       1292 |
| rarb-code                            |       1117 |       1117 |             0 |            0 |             563 |          8 |        546 |
| crumb-set-operation-entity-retrieval |        423 |        423 |             0 |            0 |             326 |          3 |         94 |
| limit                                |        344 |        344 |             0 |            0 |             317 |          0 |         27 |
| beir-nfcorpus                        |        323 |        323 |             0 |            0 |             220 |         26 |         77 |
| bright-theoremqa-questions           |        194 |        194 |             0 |            0 |              73 |          5 |        116 |
| bright-leetcode                      |        142 |        142 |             0 |            0 |              75 |         19 |         48 |
| crumb-tip-of-the-tongue              |        135 |        135 |             0 |            0 |              66 |          1 |         68 |
| crumb-clinical-trial                 |        113 |        113 |             0 |            0 |             111 |          1 |          1 |
| bright-aops                          |        111 |        111 |             0 |            0 |              61 |          0 |         50 |
| crumb-stack-exchange                 |        107 |        107 |             0 |            0 |              56 |          9 |         42 |
| crumb-paper-retrieval                |         72 |         72 |             0 |            0 |              72 |          0 |          0 |
| crumb-theorem-retrieval              |         69 |         69 |             0 |            0 |              20 |          0 |         49 |

labelled 24,338 of 50,000 | qrels_ready 1,874


| dataset                              |   labelled |   routes_differ |   decisive | decisive share   |   median margin | top route   | top share   |
|:-------------------------------------|-----------:|----------------:|-----------:|:-----------------|----------------:|:------------|:------------|
| msmarco-passage-dev                  |       7697 |            4006 |        927 | 23%              |           0     | dense_only  | 83%         |
| rarb-math                            |       6276 |            3191 |        586 | 18%              |           0.013 | dense_only  | 57%         |
| crumb-code-retrieval                 |       3665 |             903 |        189 | 21%              |           0.021 | dense_only  | 75%         |
| crumb-legal-qa                       |       3550 |            2174 |        485 | 22%              |           0.043 | dense_only  | 87%         |
| rarb-code                            |       1117 |             563 |        132 | 23%              |           0.06  | dense_only  | 98%         |
| crumb-set-operation-entity-retrieval |        423 |             326 |         53 | 16%              |           0.022 | sparse_only | 45%         |
| limit                                |        344 |             317 |        149 | 47%              |           0.045 | sparse_only | 99%         |
| beir-nfcorpus                        |        323 |             220 |         28 | 13%              |           0.011 | dense_only  | 49%         |
| bright-theoremqa-questions           |        194 |              73 |          7 | 10%              |           0.022 | dense_only  | 66%         |
| bright-leetcode                      |        142 |              75 |          4 | 5%               |           0.024 | dense_only  | 49%         |
| crumb-tip-of-the-tongue              |        135 |              66 |          3 | 5%               |           0.017 | dense_only  | 70%         |
| crumb-clinical-trial                 |        113 |             111 |         30 | 27%              |           0.033 | dense_only  | 71%         |
| bright-aops                          |        111 |              61 |          0 | 0%               |           0.016 | dense_only  | 49%         |
| crumb-stack-exchange                 |        107 |              56 |         13 | 23%              |           0.025 | dense_only  | 77%         |
| crumb-paper-retrieval                |         72 |              72 |          5 | 7%               |           0.025 | dense_only  | 61%         |
| crumb-theorem-retrieval              |         69 |              20 |          1 | 5%               |           0.017 | dense_only  | 55%         |

## 18 — How much is routing worth? The headroom readout

Three levels, same labelled rows:

1. **One global constant** — always answer with the single best method, everywhere.
2. **Best constant per collection** — someone tells you, per dataset, which one method
   works best there, and you follow it blindly.
3. **Per-query oracle** — a fortune-teller picks the best method for every individual
   query. This is the ceiling.

The 1→2 gap is what *knowing your collection* is worth; 2→3 is what *judging each query*
adds. Together they are the business case for a router.

**Read it with the fine print, always:**

- The oracle is a **ceiling, not an achievement**. Published attempts at this kind of
  per-query selection achieved ~4% at best — a router capturing even half of our ceiling
  would be exceptional.
- The pooled number depends on the **dataset mix**, which was never designed for routing:
  these queries were composed for feature diversity. **This is not an optimal routing
  dataset**, and its pooled headroom must not be quoted without this caveat.
- Judgment holes are unmeasured and bias scores against dense; every number is pinned to
  the current embedding stack (bge-small).
- **One-sided decisiveness is not headroom.** `limit` is 43% decisive with ~0 headroom:
  one method wins every decisive row there, so the constant already captures it — visible
  in the winner table below.

Decisive = the winner put a relevant document at rank 1 and the runner-up did not; the
margin threshold is derived from the objective's weights, never hand-typed.

Ratified decision and full caveat list: SPEC.md decision 44.

In [24]:
show(labels.headroom_decomposition().round(3))

headroom = labels.headroom()
show(headroom)

winners = labels.decisive_winners()
show(winners)

pooled = headroom[headroom.dataset == "POOLED"].iloc[0]
mix = headroom[headroom.dataset != "POOLED"].nlargest(1, "labelled").iloc[0]
print(f"ceiling over one global constant: +{pooled.headroom_pct}% "
      f"({pooled.constant:.3f} -> {pooled.oracle:.3f})")
print(f"lane-mix warning: {mix.dataset} alone is "
      f"{mix.labelled / pooled.labelled * 100:.0f}% of labelled rows "
      f"and its own ceiling is only +{mix.headroom_pct}%")

| level                            |   score |   gain_vs_previous_pct |
|:---------------------------------|--------:|-----------------------:|
| one global constant (dense_only) |   0.481 |                  nan   |
| best constant per collection     |   0.511 |                    6.3 |
| per-query oracle (ceiling)       |   0.557 |                    9   |

| dataset                              |   labelled |   oracle | best_constant   |   constant |   headroom |   headroom_pct |   decisive_share |   all_zero_share |
|:-------------------------------------|-----------:|---------:|:----------------|-----------:|-----------:|---------------:|-----------------:|-----------------:|
| beir-nfcorpus                        |        323 |    0.504 | pure_rrf        |      0.442 |      0.062 |           14   |            0.077 |            0.238 |
| bright-aops                          |        111 |    0.053 | pure_rrf        |      0.043 |      0.01  |           23.1 |            0     |            0.45  |
| bright-leetcode                      |        142 |    0.229 | pure_rrf        |      0.191 |      0.037 |           19.6 |            0.028 |            0.338 |
| bright-theoremqa-questions           |        194 |    0.114 | dense_only      |      0.094 |      0.02  |           21.4 |            0.036 |            0.598 |
| crumb-clinical-trial                 |        113 |    0.817 | dense_only      |      0.731 |      0.086 |           11.8 |            0.159 |            0.009 |
| crumb-code-retrieval                 |       3665 |    0.111 | dense_only      |      0.098 |      0.013 |           13.6 |            0.041 |            0.747 |
| crumb-legal-qa                       |       3550 |    0.31  | dense_only      |      0.285 |      0.025 |            8.9 |            0.126 |            0.364 |
| crumb-paper-retrieval                |         72 |    0.869 | dense_only      |      0.829 |      0.04  |            4.8 |            0.056 |            0     |
| crumb-set-operation-entity-retrieval |        423 |    0.478 | pure_rrf        |      0.393 |      0.085 |           21.8 |            0.111 |            0.222 |
| crumb-stack-exchange                 |        107 |    0.318 | dense_only      |      0.277 |      0.04  |           14.6 |            0.112 |            0.393 |
| crumb-theorem-retrieval              |         69 |    0.065 | sparse_only     |      0.05  |      0.014 |           28.7 |            0.014 |            0.71  |
| crumb-tip-of-the-tongue              |        135 |    0.227 | pure_rrf        |      0.209 |      0.019 |            8.9 |            0.015 |            0.504 |
| limit                                |        344 |    0.874 | sparse_only     |      0.871 |      0.003 |            0.3 |            0.433 |            0.078 |
| msmarco-passage-dev                  |       7697 |    0.856 | dense_only      |      0.803 |      0.053 |            6.6 |            0.12  |            0.02  |
| rarb-code                            |       1117 |    0.251 | dense_only      |      0.247 |      0.004 |            1.5 |            0.118 |            0.489 |
| rarb-math                            |       6276 |    0.676 | pure_rrf        |      0.599 |      0.077 |           12.8 |            0.093 |            0.154 |
| POOLED                               |      24338 |    0.557 | dense_only      |      0.481 |      0.076 |           15.8 |            0.103 |            0.258 |

| dataset                              |   dense_only |   pure_rrf |   sparse_only |
|:-------------------------------------|-------------:|-----------:|--------------:|
| beir-nfcorpus                        |           15 |          1 |             9 |
| bright-leetcode                      |            3 |          0 |             1 |
| bright-theoremqa-questions           |            5 |          0 |             2 |
| crumb-clinical-trial                 |           15 |          0 |             3 |
| crumb-code-retrieval                 |          126 |          5 |            21 |
| crumb-legal-qa                       |          413 |         18 |            15 |
| crumb-paper-retrieval                |            3 |          0 |             1 |
| crumb-set-operation-entity-retrieval |           22 |          4 |            21 |
| crumb-stack-exchange                 |            9 |          0 |             3 |
| crumb-theorem-retrieval              |            0 |          0 |             1 |
| crumb-tip-of-the-tongue              |            2 |          0 |             0 |
| limit                                |            0 |          0 |           149 |
| msmarco-passage-dev                  |          782 |         54 |            89 |
| rarb-code                            |          129 |          2 |             1 |
| rarb-math                            |          334 |         61 |           191 |
| POOLED                               |         1858 |        145 |           507 |

ceiling over one global constant: +15.8% (0.481 -> 0.557)
lane-mix warning: msmarco-passage-dev alone is 32% of labelled rows and its own ceiling is only +6.6%
